# Property Listing Content Generator — Eval-First

The deliverable here is a **validated evaluator**, not the generator. This notebook runs **fully offline (zero API calls)**: it re-executes the deterministic evaluation live and reads the LLM-judged results from committed `.eval` logs.

Two clearly-separated kinds of result below:
1. **Committed real runs** — genuine Anthropic generations + judge scores, read from `logs/`.
2. **Live offline recompute** — deterministic scorers re-run here via a replay solver, proving reproducibility without a key.

View the raw logs interactively with: `uv run inspect view --log-dir logs`

In [1]:
import json
from content_pipeline.report import find_log, gold_report

def show_metrics(log):
    for s in log.results.scores:
        print(f"  {s.name:24s} " + ", ".join(f"{k}={v.value:.2f}" for k, v in s.metrics.items()))

## 1. A generated listing (committed real run)
Structured, grounded copy. Note each highlight carries the `source_field` it derives from — that binding makes citation grounding a deterministic check.

In [2]:
print(json.dumps(json.load(open('fixtures/generations/villa.json')), indent=2))

{
  "hero_headline": "Seafront Villa with Private Pool Steps from the Marina in Sitges",
  "highlights": [
    {
      "text": "Sleeps up to 6 guests across 3 bedrooms and 2 bathrooms.",
      "source_field": "rental_info"
    },
    {
      "text": "Rated 4.96 out of 5 across 87 guest reviews.",
      "source_field": "average_review_score"
    },
    {
      "text": "Guests rave about the stunning sea views, spotless private pool, and easy walk to the old town.",
      "source_field": "reviews"
    },
    {
      "text": "Located near the marina in Sitges, Spain, with the old town just a ten-minute stroll away.",
      "source_field": "description"
    },
    {
      "text": "Flexible booking: full refund available up to 30 days before arrival.",
      "source_field": "policies"
    }
  ],
  "about_section": "Casa del Mar is a bright three-bedroom villa in Sitges, Spain, set steps from the marina with beautiful sea views. The villa features a private pool and is within a ten-minute wa

## 2. Production scorers on the real generations (committed)
Tier 1 deterministic (citation validity, quality) + Tier 2 LLM judge: atomic-claim **faithfulness** (precision — is every claim grounded?), **coverage** (recall — does the copy surface the marquee selling points?), and brand voice. These numbers come from the committed `generation_eval` log. `judge_value_mean` excludes any unparseable-judge samples (`judge_error_rate`), so a flaky judge never reads as `faithfulness=0`.

In [3]:
gen = find_log('generation_eval')
print('generation_eval (real Anthropic run):')
show_metrics(gen)

generation_eval (real Anthropic run):
  citation_validity        accuracy=1.00, stderr=0.00
  quality                  accuracy=0.00, stderr=0.00
  faithfulness             judge_value_mean=0.90, judge_error_rate=0.00, stderr=0.10
  coverage                 judge_value_mean=1.00, judge_error_rate=0.00, stderr=0.00
  model_graded_qa          accuracy=0.50, stderr=0.00


## 3. META-EVAL — validating the evaluator (the point of the submission)
A suite that scores everything "pass" is worthless. We prove the scorers (a) **catch real failures** (recall) and (b) **don't cry wolf** on good copy (false-positive rate), over hand-labeled good/bad generations.

### 3a. Deterministic tier — recomputed LIVE, zero API
Re-run here via the replay solver + `mockllm` (no key). These match the committed deterministic scores bit-for-bit.

In [4]:
import os, tempfile

# inspect_ai's sync eval() calls asyncio.run(), which crashes inside a notebook
# kernel's already-running event loop (VS Code: "Already running asyncio in this
# thread"). eval_async + top-level await runs cleanly in VS Code, Jupyter, and
# nbconvert. INSPECT_DISPLAY=none suppresses the live progress widget.
os.environ["INSPECT_DISPLAY"] = "none"
from inspect_ai import eval_async
from content_pipeline.eval_task import meta_eval

with tempfile.TemporaryDirectory() as d:
    live = (await eval_async(meta_eval(include_judge=False), model="mockllm/model", log_dir=d))[0]
print("meta_eval deterministic (live, offline):")
show_metrics(live)

---------------------------------------------------------                                                          
meta_eval (10 samples): mockllm/model                                                                              
system_message: You write marketing copy for vacation-rental..., max_tokens: 2048, temperature: 0.0, include_judge:
False, dataset: (samples)                                                                                          
---------------------------------------------------------

replay_ge... | Steps:  10/10 100% | Samples:  10/ 10 | content_pipeline/detection_rate:  n/a |  | HTTP retries: 0



---------------------------------------------------------                                                          
meta_eval (10 samples): mockllm/model                                                                              
system_message: You write marketing copy for vacation-rental..., max_tokens: 2048, temperature: 0.0, include_judge:
False, dataset: (samples)                                                                                          
                                                                                                                   
total time:                                                       0:00:00                                          
meta_citation_validity         meta_quality                                                                        
detection_rate          1.000  detection_rate       0.500                                                          
false_positive_rate     0.000  false_positive_rate  0.000                                                          
Log:                                                                                                               
../../../../../var/folders/w9/lqtl31zs7q5_f1zf9qqj47b00000gn/T/tmpkognp0ch/2026-06-17T19-27-03-00-00_meta-eval_6Lc…
---------------------------------------------------------

meta_eval deterministic (live, offline):
  meta_citation_validity   detection_rate=1.00, false_positive_rate=0.00
  meta_quality             detection_rate=0.50, false_positive_rate=0.00


### 3b. Judge tier — GATED by detection + false-positive metrics (committed)
The judge isn't just *reported*, it's *validated*: `meta_faithfulness_scorer` carries `judge_detection_rate` (per-claim recall on planted hallucinations — did it flag the *specific* fabricated claim?) and `judge_false_positive_rate` (over-flagging on genuinely-clean copy). The judge needs a key, so verdicts are read from the committed `meta_eval` log. Per-sample faithfulness below should be low on the planted hallucinations and high on the good fixtures.

In [5]:
meta = find_log('meta_eval')
print(f"{'sample':26s} {'label':5s} {'judge_target':12s} faithfulness")
for s in sorted(meta.samples, key=lambda s: s.id):
    sc = s.scores.get('faithfulness')
    if sc:
        tag = 'hallucination' if s.metadata['judge_failure'] else ''
        print(f"  {s.id:26s} {s.metadata['label']:5s} {tag:12s} {sc.value:.2f}")

sample                     label judge_target faithfulness
  bad_bland                  bad                0.25
  bad_empty_citation         bad                0.83
  bad_fabricated_landmark    bad   hallucination 0.50
  bad_incomplete_villa       bad   hallucination 0.86
  bad_novel_bland            bad                0.62
  bad_plausible_hallucination bad   hallucination 0.50
  bad_unlisted_amenity       bad                0.57
  bad_wrong_number           bad   hallucination 0.50
  good_cottage               good               1.00
  good_villa                 good               0.78


In [6]:
# The judge is GATED, not just reported: meta_eval scores it with per-claim
# detection + false-positive metrics (the same first-class treatment the
# deterministic tier gets). These come from the committed meta_eval log.
print("meta_eval judge metrics (committed):")
show_metrics(meta)

meta_eval judge metrics (committed):
  meta_citation_validity   detection_rate=1.00, false_positive_rate=0.00
  meta_quality             detection_rate=0.50, false_positive_rate=0.00
  faithfulness             judge_detection_rate=1.00, judge_false_positive_rate=0.13, judge_error_rate=0.00
  coverage                 judge_value_mean=0.77, judge_error_rate=0.00, stderr=0.06


### 3c. Judge vs human — confusion matrix (committed verdicts vs gold labels)
Positive class = *not grounded* (a hallucination the judge should flag). **Read this as methodology, not a trustworthy point estimate** — n is tiny and there is a single annotator (see caveats).

In [7]:
r = gold_report()
# Cohen's kappa (chance-corrected) + TPR/TNR — not raw agreement, which flatters
# under class imbalance. Tiny n: read as methodology, CI is very wide.
print(f"n={len(r.rows)}   kappa={r.cohen_kappa:.2f}   TPR(recall)={r.tpr:.0%}   "
      f"TNR={r.tnr:.0%}   precision={r.precision:.0%}   (raw agreement={r.agreement:.0%})\n")
print('                 judge:flag   judge:ground')
print(f"human:not-ground   TP={r.tp:<8d}   FN={r.fn}")
print(f"human:grounded     FP={r.fp:<8d}   TN={r.tn}\n")
print('Disagreements (judge over-flagged legitimate claims):')
for row in r.rows:
    if not row.agree:
        print(f"  judge={row.judge_verdict!r}: {row.claim}")

n=12   kappa=0.53   TPR(recall)=100%   TNR=62%   precision=57%   (raw agreement=75%)

                 judge:flag   judge:ground
human:not-ground   TP=4          FN=0
human:grounded     FP=3          TN=5

Disagreements (judge over-flagged legitimate claims):
  judge=None: Enjoy your own private pool, perfect for cooling off in the Spanish sun.
  judge=None: Warm up beside a classic fireplace after a day of exploring.
  judge='unsupported': Air conditioning throughout for warm summers.


## 4. Reading
- **Deterministic tier**: citation validity catches every designed negative (100% / 0% FP). `quality` detection is **50%, by design** — a *held-out* bland negative using phrasing **not** on the denylist slips past, honestly exposing that a 7-phrase blocklist doesn't generalize. The judge brand-voice backstops it. (On the real generations, `quality` also fired on genuine repetition — "the old town" / "in the Cotswolds" ×3.)
- **Judge tier is GATED, not just reported**: `judge_detection_rate` = 100% (flags the *specific* planted claim, incl. a prompt-injection attempt), `judge_false_positive_rate` ≈ 13% (it over-flags grounded amenity flourishes), `judge_error_rate` = 0%.
- **Judge vs human**: Cohen's **κ ≈ 0.53** (moderate) — the honest number; raw agreement (75%) flatters under class imbalance. TPR 100%, TNR ~63%. The over-flags are real calibration debt, surfaced not hidden.
- **Coverage (recall)**: both real generations surface 100% of marquee facts; `BAD_INCOMPLETE_VILLA` is the faithful-but-low-coverage case that *only* the coverage dimension catches.
- The eval also caught a **bug in my own code**: an incomplete grounding corpus made the judge mark valid claims unsupported (good-copy faithfulness ~0.77). Fixing the corpus restored it. That is the eval-first loop working.

### Honest caveats (also in README)
- n is a demonstration size; treat every rate (κ included — its CI is huge here) as methodology that scales to hundreds, not significant estimates.
- Single annotator who also wrote the judge prompt — no inter-annotator agreement.
- The gold set validates judge *verdicts*; the upstream *decomposition* is validated separately (`tests/test_decomposition.py`).
- **Documented, not built** (next steps): a judge model ≠ the generator (self-enhancement bias); meta-validating the brand-voice scorer; structurally separating trusted facts from untrusted review text (injection); regression gating in CI; multilingual; image grounding.